## Concatenate

### Zadanie 1 - łączenie raportów z wielu miesięcy


Dane przychodzą z 3 systemów sprzedażowych
Każdy system może powtórzyć request - powstają duplikaty eventów *błędna architektura API, trzeba je usunąć

ZADANIE:

1. Połącz wszystkie źródła danych.
2. Dodaj kolumnę 'zrodlo'.
3. Sprawdź liczbę rekordów przed czyszczeniem.
4. Usuń duplikaty, definiując rekord jako:
   (timestamp, klient, produkt, wartosc)
5. Sprawdź liczbę rekordów po czyszczeniu.
6. Wyświetl wynik.

In [2]:
import pandas as pd

online = pd.DataFrame({
    'timestamp': ['2024-01-01 10:00', '2024-01-01 10:05', '2024-01-01 10:10'],
    'klient': ['Jan', 'Anna', 'Piotr'],
    'produkt': ['Laptop', 'Mysz', 'Klawiatura'],
    'wartosc': [4000, 80, 250]
})

store = pd.DataFrame({
    'timestamp': ['2024-01-01 10:10', '2024-01-01 10:15', '2024-01-01 10:20'],
    'klient': ['Piotr', 'Adam', 'Kasia'],
    'produkt': ['Klawiatura', 'Monitor', 'Laptop'],
    'wartosc': [250, 1200, 4200]
})

marketplace = pd.DataFrame({
    'timestamp': ['2024-01-01 10:20', '2024-01-01 10:25', '2024-01-01 10:25'],
    'klient': ['Kasia', 'Tomek', 'Tomek'],  # retry API → duplikat eventu
    'produkt': ['Laptop', 'Słuchawki', 'Słuchawki'],
    'wartosc': [4200, 300, 300]
})

marketplace_invalid = pd.concat([marketplace, marketplace], ignore_index=True)

print(marketplace_invalid)

          timestamp klient    produkt  wartosc
0  2024-01-01 10:20  Kasia     Laptop     4200
1  2024-01-01 10:25  Tomek  Słuchawki      300
2  2024-01-01 10:25  Tomek  Słuchawki      300
3  2024-01-01 10:20  Kasia     Laptop     4200
4  2024-01-01 10:25  Tomek  Słuchawki      300
5  2024-01-01 10:25  Tomek  Słuchawki      300


In [12]:
final_df = (
    pd.concat([
        online,
        store,
        marketplace_invalid
    ], ignore_index=True)
    # .sort_values('klient')
    .drop_duplicates(
        subset=['timestamp', 'klient', 'produkt', 'wartosc']
    )
)

final_df

,timestamp,klient,produkt,wartosc
0,2024-01-01 10:00,Jan,Laptop,4000
1,2024-01-01 10:05,Anna,Mysz,80
2,2024-01-01 10:10,Piotr,Klawiatura,250
4,2024-01-01 10:15,Adam,Monitor,1200
5,2024-01-01 10:20,Kasia,Laptop,4200
7,2024-01-01 10:25,Tomek,Słuchawki,300


### Zadanie 2 - concat z różnymi kolumnami i diagnostyka braków

Trzy systemy eksportują dane o pracownikach, ale każdy ma nieco inne kolumny. 

Połącz je w jeden DataFrame (outer i inner), zbadaj które kolumny brakują z których systemów, 

uzupełnij braki sensownymi wartościami domyślnymi i dodaj kolumnę 'zrodlo'.

In [13]:
import pandas as pd

system_hr = pd.DataFrame({
    'emp_id': [1, 2, 3],
    'imie': ['Anna', 'Bartek', 'Celina'],
    'dzial': ['IT', 'HR', 'IT'],
    'etat': [1.0, 0.5, 1.0]
})

system_place = pd.DataFrame({
    'emp_id': [2, 3, 4],
    'imie': ['Bartek', 'Celina', 'Dawid'],
    'wynagrodzenie': [6000, 5500, 7000],
    'waluta': ['PLN', 'PLN', 'PLN']
})

system_oceny = pd.DataFrame({
    'emp_id': [1, 4, 5],
    'imie': ['Anna', 'Dawid', 'Ewa'],
    'ocena_roczna': [4.5, 3.8, 4.2],
    'czy_senior': [True, False, True]
})

In [15]:
df_hr = system_hr.assign(source='HR')
df_pl = system_place.assign(source='place')
df_sys = system_oceny.assign(source='oceny')

# outer join
outer = pd.concat([df_hr, df_pl, df_sys], ignore_index=True, join='outer')

# innej join
inner = pd.concat([df_hr, df_pl, df_sys], ignore_index=True, join='inner')

display('Ramka outer:')
display(outer)

display('Ramka inner:')
display(inner)

'Ramka outer:'

,emp_id,imie,dzial,etat,source,wynagrodzenie,waluta,ocena_roczna,czy_senior
0,1,Anna,IT,1.0,HR,NaN,NaN,NaN,NaN
1,2,Bartek,HR,0.5,HR,NaN,NaN,NaN,NaN
2,3,Celina,IT,1.0,HR,NaN,NaN,NaN,NaN
3,2,Bartek,NaN,NaN,place,6000.0,PLN,NaN,NaN
4,3,Celina,NaN,NaN,place,5500.0,PLN,NaN,NaN
5,4,Dawid,NaN,NaN,place,7000.0,PLN,NaN,NaN
6,1,Anna,NaN,NaN,oceny,NaN,NaN,4.5,True
7,4,Dawid,NaN,NaN,oceny,NaN,NaN,3.8,False
8,5,Ewa,NaN,NaN,oceny,NaN,NaN,4.2,True


'Ramka inner:'

,emp_id,imie,source
0,1,Anna,HR
1,2,Bartek,HR
2,3,Celina,HR
3,2,Bartek,place
4,3,Celina,place
5,4,Dawid,place
6,1,Anna,oceny
7,4,Dawid,oceny
8,5,Ewa,oceny


In [16]:
display(
    outer.groupby('source').apply(
        lambda x: x.isna().sum(),
        include_groups=False
    )
)

,emp_id,imie,dzial,etat,wynagrodzenie,waluta,ocena_roczna,czy_senior
source,,,,,,,,
HR,0,0,0,0,3,3,3,3
oceny,0,0,3,3,3,3,0,0
place,0,0,3,3,0,0,3,3


In [20]:
# uzupełnianie braków:

outer_filled = outer.copy()

outer_filled['wynagrodzenie'] = outer_filled['wynagrodzenie'].fillna(0)
outer_filled['waluta'] = outer_filled['waluta'].fillna('PLN')
outer_filled['etat'] = outer_filled['etat'].fillna(1.0)

outer_filled

,emp_id,imie,dzial,etat,source,wynagrodzenie,waluta,ocena_roczna,czy_senior
0,1,Anna,IT,1.0,HR,0.0,PLN,NaN,NaN
1,2,Bartek,HR,0.5,HR,0.0,PLN,NaN,NaN
2,3,Celina,IT,1.0,HR,0.0,PLN,NaN,NaN
3,2,Bartek,NaN,1.0,place,6000.0,PLN,NaN,NaN
4,3,Celina,NaN,1.0,place,5500.0,PLN,NaN,NaN
5,4,Dawid,NaN,1.0,place,7000.0,PLN,NaN,NaN
6,1,Anna,NaN,1.0,oceny,0.0,PLN,4.5,True
7,4,Dawid,NaN,1.0,oceny,0.0,PLN,3.8,False
8,5,Ewa,NaN,1.0,oceny,0.0,PLN,4.2,True



---

### Merge

In [26]:
import pandas as pd

df1 = pd.DataFrame({ 'key': ['A', 'B'], 'x': [1, 2]})

df2 = pd.DataFrame({ 'key': ['B', 'C'], 'y': [10, 20]})

display(df1)
display(df2)

,key,x
0,A,1
1,B,2


,key,y
0,B,10
1,C,20


>> Concat

In [25]:
concat_outer = pd.concat([df1, df2], join='outer', ignore_index=True)
concat_inner = pd.concat([df1, df2], join='inner', ignore_index=True)

display("Outer concat: ", concat_outer)
display("Inner concat: ", concat_inner)

'Outer concat: '

,key,x,y
0,A,1.0,NaN
1,B,2.0,NaN
2,B,NaN,10.0
3,C,NaN,20.0


'Inner concat: '

,key
0,A
1,B
2,B
3,C


>> Merge

In [31]:
outer_merge = df1.merge(
    df2, 
    on='key',
    how='outer'
)
inner_merge = df1.merge(
    df2, 
    on='key',
    how='inner'
)

left_merge = df1.merge(
    df2, 
    on='key',
    how='left'
)

right_merge = df1.merge(
    df2, 
    on='key',
    how='right'
)


In [32]:
display(df1)
display(df2)

display("Outer concat: ", concat_outer)
display("Inner concat: ", concat_inner)
display("Outer merge: ",outer_merge)
display("Inner merge: ",inner_merge)
display("left merge: ",left_merge)
display("right merge: ",right_merge)

,key,x
0,A,1
1,B,2


,key,y
0,B,10
1,C,20


'Outer concat: '

,key,x,y
0,A,1.0,NaN
1,B,2.0,NaN
2,B,NaN,10.0
3,C,NaN,20.0


'Inner concat: '

...

,key,x,y
0,B,2,10


'left merge: '

,key,x,y
0,A,1,NaN
1,B,2,10.0


'right merge: '

,key,x,y
0,B,2.0,10
1,C,NaN,20



---
### JOIN



result = df1.join(df2, how=?)

#### Zadanie 1 - join na indeksie z różnymi strategiami

Masz tabelę produktów i tabelę z danymi magazynowymi. 

Połącz je metodą join() (na indeksie). 

Produkt_id jest indeksem obu tabel. 

Porównaj wyniki dla how='left', 'right', 'inner', 'outer' i wyjaśnij różnice.

In [33]:
import pandas as pd
import numpy as np

produkty = pd.DataFrame({
    'nazwa': ['Laptop', 'Mysz', 'Monitor', 'Klawiatura', 'Słuchawki'],
    'cena': [3500, 120, 900, 250, 400],
    'kategoria': ['PC', 'Akcesoria', 'PC', 'Akcesoria', 'Audio']
}, index=[101, 102, 103, 104, 105])

produkty.index.name = 'produkt_id'  # nazwany index

magazyn = pd.DataFrame({
    'stan': [15, 0, 8, 32, 5],
    'lokalizacja': ['A1', 'B3', 'A2', 'C1', 'B1'],
    'ostatnia_dostawa': ['2024-03-01', '2024-01-15', '2024-03-10', '2024-02-20', '2024-03-05']
}, index=[101, 102, 103, 106, 107])

magazyn.index.name = 'produkt_id'

In [34]:
display(produkty, magazyn)

,nazwa,cena,kategoria
produkt_id,,,
101,Laptop,3500,PC
102,Mysz,120,Akcesoria
103,Monitor,900,PC
104,Klawiatura,250,Akcesoria
105,Słuchawki,400,Audio


,stan,lokalizacja,ostatnia_dostawa
produkt_id,,,
101,15,A1,2024-03-01
102,0,B3,2024-01-15
103,8,A2,2024-03-10
106,32,C1,2024-02-20
107,5,B1,2024-03-05


In [36]:
display('========== LEFT JOIN (wszystkie wiersze z lewej ramki + pasujące z prawej) ==========')
left_join = produkty.join(magazyn, how='left')
display(left_join)

display('========== RIGT JOIN (wszystkie wiersze z prawej ramki + pasujące z lewej)==========')
right_join = produkty.join(magazyn, how='right')
display(right_join)

display('========== WSZYSTKIE REKORDY ==========')
outer_join = produkty.join(magazyn, how="outer")
display(outer_join)

display('========== WSPÓLNE REKORDY ==========')
inner_join = produkty.join(magazyn, how="inner")
display(inner_join)

'========== LEFT JOIN (wszystkie wiersze z lewej ramki + pasujące z prawej) =========='

,nazwa,cena,kategoria,stan,lokalizacja,ostatnia_dostawa
produkt_id,,,,,,
101,Laptop,3500,PC,15.0,A1,2024-03-01
102,Mysz,120,Akcesoria,0.0,B3,2024-01-15
103,Monitor,900,PC,8.0,A2,2024-03-10
104,Klawiatura,250,Akcesoria,NaN,NaN,NaN
105,Słuchawki,400,Audio,NaN,NaN,NaN


'========== RIGT JOIN (wszystkie wiersze z prawej ramki + pasujące z lewej)=========='

,nazwa,cena,kategoria,stan,lokalizacja,ostatnia_dostawa
produkt_id,,,,,,
101,Laptop,3500.0,PC,15,A1,2024-03-01
102,Mysz,120.0,Akcesoria,0,B3,2024-01-15
103,Monitor,900.0,PC,8,A2,2024-03-10
106,NaN,NaN,NaN,32,C1,2024-02-20
107,NaN,NaN,NaN,5,B1,2024-03-05


'========== WSZYSTKIE REKORDY =========='

,nazwa,cena,kategoria,stan,lokalizacja,ostatnia_dostawa
produkt_id,,,,,,
101,Laptop,3500.0,PC,15.0,A1,2024-03-01
102,Mysz,120.0,Akcesoria,0.0,B3,2024-01-15
103,Monitor,900.0,PC,8.0,A2,2024-03-10
104,Klawiatura,250.0,Akcesoria,NaN,NaN,NaN
105,Słuchawki,400.0,Audio,NaN,NaN,NaN
106,NaN,NaN,NaN,32.0,C1,2024-02-20
107,NaN,NaN,NaN,5.0,B1,2024-03-05


'========== WSPÓLNE REKORDY =========='

,nazwa,cena,kategoria,stan,lokalizacja,ostatnia_dostawa
produkt_id,,,,,,
101,Laptop,3500,PC,15,A1,2024-03-01
102,Mysz,120,Akcesoria,0,B3,2024-01-15
103,Monitor,900,PC,8,A2,2024-03-10


In [40]:
# analizujemy left_join
left_missing = left_join[left_join['stan'].isna()]
print('Rekordy bez wpisu w "magazynie"')
print(left_missing)

# analizujemy right_join
right_missing = right_join[right_join['cena'].isna()]
print('Rekordy magazynowe bez wpisu w "produktach"')
print(right_missing)

Rekordy bez wpisu w "magazynie"
                 nazwa  cena  kategoria  stan lokalizacja ostatnia_dostawa
produkt_id                                                                
104         Klawiatura   250  Akcesoria   NaN         NaN              NaN
105          Słuchawki   400      Audio   NaN         NaN              NaN
Rekordy magazynowe bez wpisu w "produktach"
           nazwa  cena kategoria  stan lokalizacja ostatnia_dostawa
produkt_id                                                         
106          NaN   NaN       NaN    32          C1       2024-02-20
107          NaN   NaN       NaN     5          B1       2024-03-05



----
### UWAGA! Eksplozja wierszy

In [ ]:
# EKPLOZJA WIERSZY!

import pandas as pd

# Lewa ramka: Promocje w sklepach
df_promocje = pd.DataFrame({
    'sklep_id': [101, 101, 101],
    'promocja': ['Gratis', 'Rabat 20%', 'Darmowa dostawa']
})

# Prawa ramka: Pracownicy w sklepach
df_pracownicy = pd.DataFrame({
    'sklep_id': [101, 101, 101],
    'pracownik': ['Anna', 'Jan', 'Marek']
})

df_pracownicy_unique = pd.DataFrame({
    'sklep_id': [101, 102, 103],
    'pracownik': ['Anna', 'Jan', 'Marek']
})

# Łączenie ramek (wiele do wielu)
wynik = pd.merge(df_promocje, df_pracownicy, on='sklep_id')
print(wynik)


   sklep_id         promocja pracownik
0       101           Gratis      Anna
1       101           Gratis       Jan
2       101           Gratis     Marek
3       101        Rabat 20%      Anna
4       101        Rabat 20%       Jan
5       101        Rabat 20%     Marek
6       101  Darmowa dostawa      Anna
7       101  Darmowa dostawa       Jan
8       101  Darmowa dostawa     Marek


In [43]:
from pandas.errors import MergeError

# Wymuszamy relację many_to_one
try:
    wynik_bezpieczny = pd.merge(
        df_promocje, 
        df_pracownicy, 
        on='sklep_id', 
        validate='many_to_one' # one_to_one | one_to_many | many_to_one | many_to_many
    )
except MergeError as e:
    print(f"Błąd! Wykryto eksplozję danych: {e}")

Błąd! Wykryto eksplozję danych: Merge keys are not unique in right dataset; not a many-to-one merge


In [47]:
from pandas.errors import MergeError

# Wymuszamy relację many_to_one
try:
    wynik_bezpieczny = pd.merge(
        df_promocje, 
        df_pracownicy_unique, 
        on='sklep_id',
        how="outer",
        validate='many_to_one' # one_to_one | one_to_many | many_to_one | many_to_many
    )
except MergeError as e:
    print(f"Błąd! Wykryto eksplozję danych: {e}")

display(wynik_bezpieczny)

,sklep_id,promocja,pracownik
0,101,Gratis,Anna
1,101,Rabat 20%,Anna
2,101,Darmowa dostawa,Anna
3,102,NaN,Jan
4,103,NaN,Marek
